In [ ]:
# ============================================================
# Descriptive Statistics Report Generator
# ============================================================
# Purpose:
#   Generate descriptive statistics for HRS variables across
#   multiple waves with detailed breakdowns.
# ============================================================

from pyspark.sql import functions as F
import pandas as pd

# ============================================================
# INPUT PARAMETERS
# ============================================================
catalog_name = "staging_catalog"             # UC catalog name
schema_name = "slv_cdm_hrs"                  # UC schema name

# =============List of DEMOGRAPHICS variables to analyze======
table_name = "hrs_demographics"              # Table to analyze
variable_names = ["agey_e", "cenreg", "mstat"]

# =============List of HEALTH variables to analyze============
#table_name = "hrs_health"              # Table to analyze
#variable_names = ["shlt", "bmi", "hibpe", "diabe", "cancre", "lunge", "hearte", "stroke", "psyche", "arthre"]

# =============List of LEAVE_BEHIND variables to analyze======
#table_name = "hrs_leave_behind"              # Table to analyze
#variable_names = ["lbneur", "lbext", "lbopen", "lbagr", "lbcon5"]

wave_numbers = [1, 2, 3, 4, 5]               # List of waves to include

# ============================================================
# DATA RETRIEVAL
# ============================================================

# Construct fully qualified table name
full_table_name = f"{catalog_name}.{schema_name}.{table_name}"

print(f"Analyzing variables: {', '.join(variable_names)}")
print(f"From table: {full_table_name}")
print(f"Waves: {wave_numbers}")
print("="*60)

# Read the table
df = spark.table(full_table_name)

# Verify wave_number column exists
if "wave_number" not in df.columns:
    raise ValueError("Table must have 'wave_number' column.")

# Filter for specified waves
df_filtered = df.filter(F.col("wave_number").isin(wave_numbers))

# ============================================================
# PROCESS EACH VARIABLE
# ============================================================

for idx, variable_name in enumerate(variable_names, 1):
    
    # Check if variable exists in the table
    if variable_name not in df.columns:
        print(f"\n⚠️  WARNING: Variable '{variable_name}' not found in table. Skipping.")
        print("="*60)
        continue
    
    # ============================================================
    # CALCULATE STATISTICS BY WAVE
    # ============================================================
    
    stats_by_wave = df_filtered.groupBy("wave_number").agg(
        F.count(F.col(variable_name)).alias("N"),
        F.mean(F.col(variable_name)).alias("Mean"),
        F.stddev(F.col(variable_name)).alias("STD"),
        F.min(F.col(variable_name)).alias("Min"),
        F.max(F.col(variable_name)).alias("Max")
    ).orderBy("wave_number")
    
    # Convert to pandas for better display formatting
    stats_df = stats_by_wave.toPandas()
    
    # ============================================================
    # CALCULATE OVERALL STATISTICS (ALL WAVES COMBINED)
    # ============================================================
    
    overall_stats = df_filtered.agg(
        F.count(F.col(variable_name)).alias("N"),
        F.mean(F.col(variable_name)).alias("Mean"),
        F.stddev(F.col(variable_name)).alias("STD"),
        F.min(F.col(variable_name)).alias("Min"),
        F.max(F.col(variable_name)).alias("Max")
    ).toPandas()
    
    # Add wave column for overall stats
    overall_stats.insert(0, "wave_number", "Overall")
    
    # ============================================================
    # COMBINE AND FORMAT RESULTS
    # ============================================================
    
    # Combine wave-specific and overall statistics
    final_stats = pd.concat([stats_df, overall_stats], ignore_index=True)
    
    # Format numeric columns to 2 decimal places
    for col in ["Min", "Max", "Mean", "STD"]:
        final_stats[col] = final_stats[col].round(2)
    
    # Rename wave column for clarity
    final_stats.rename(columns={"wave_number": "Wave"}, inplace=True)
    
    # ============================================================
    # DISPLAY RESULTS
    # ============================================================
    
    print(f"\n{'='*60}")
    print(f"SECTION {idx}: {variable_name}")
    print(f"{'='*60}")
    
    display(final_stats)

print(f"\n{'='*60}")
print(f"Report Complete: {len(variable_names)} variable(s) analyzed")
print(f"{'='*60}")